# FraudShield AI: DistilBERT fine-tuning (Colab)

Runtime → Change runtime type → **T4 GPU**. This rebuilds the message splits from the raw datasets (the same seed gives the same split hashes as locally), fine-tunes DistilBERT, and saves the model to Google Drive.

If the repo is private, upload a zip of the project instead of cloning, or use a GitHub token stored in Colab Secrets.

In [ ]:
!nvidia-smi -L

In [ ]:
REPO = 'https://github.com/Moeez-abid/FraudShieldAI.git'
!git clone -q $REPO
%cd FraudShieldAI

In [ ]:
!pip -q install -r requirements.txt -r requirements-dl.txt -e .

In [ ]:
# Download, clean, deduplicate and split the messages (small, under a minute)
!python -m fraudshield.data.download --only uci_sms mendeley_smishing
!python -m fraudshield.data.clean_messages
!python -m fraudshield.data.split messages

In [ ]:
# Check the split matches the committed manifest
import json
m = json.load(open('data/processed/messages_split_manifest.json'))
{k: (v['rows'], v['hash']) for k, v in m['splits'].items()}

In [ ]:
# Single run with config defaults (~3 min on T4). Add --sweep for the lr x max_length grid (~20 min).
!python -m fraudshield.models.distilbert

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/fraudshield
!cp -r models/distilbert /content/drive/MyDrive/fraudshield/
!cp reports/metrics/distilbert_*.json reports/figures/cm_distilbert_*.png /content/drive/MyDrive/fraudshield/
print('Copy models/distilbert and the metrics JSON back into the repo (models/ is git-ignored).')